# Dubai Real Estate Project Unit Summary - Complete Data Pipeline
## Version 2025 - Comprehensive Data Processing & Enrichment

---

## 📋 Table of Contents
1. [Project Overview](#overview)
2. [Data Sources & Files](#data-sources)
3. [Pipeline Architecture](#architecture)
4. [Key Features](#features)
5. [Dependencies & Setup](#setup)
6. [Execution Guide](#execution)
7. [Output Files](#outputs)
8. [Quick Start](#quickstart)

---

## <a id="overview"></a>🎯 Project Overview

This notebook implements a **complete end-to-end data processing pipeline** for Dubai real estate projects. It:

- **Aggregates** unit-level data into project-level summaries
- **Enriches** projects data with comprehensive unit statistics
- **Categorizes** properties by bedroom type, sub-type, and location
- **Calculates** cumulative areas and carpet area metrics

**Target Users**: Real estate analysts, data scientists, portfolio managers

**Output**: Comprehensive Excel workbooks with multi-dimensional project analysis

**Last Updated**: February 2025

---

## <a id="data-sources"></a>📂 Data Sources & Input Files

| File | Path | Purpose | Key Columns |
|------|------|---------|------------|
| **Projects.csv** | `D:\Dubai\Data\Projects.csv` | Base project metadata | `project_id`, dates, location |
| **Units.csv** | `Units_with_actual_area_sqft.csv` | Unit-level details | `project_id`, `rooms_en`, `actual_area_sqft`, `property_sub_type_en` |
| **Buildings.csv** | `D:\Dubai\Data\Buildings.csv` | Building floor information | `project_id`, `floors` |

---

## <a id="quickstart"></a>⚡ Quick Start (4 Steps)

### 1. **Verify File Paths**
Open first code cell (Cell 2) and confirm:
```python
projects_file = r"D:\Dubai\Data\Projects.csv"           # ✓ Correct?
units_file    = r"D:\Dubai\...Units_with_actual_area_sqft.csv"  # ✓ Correct?
buildings_file = r"D:\Dubai\Data\Buildings.csv"         # ✓ Correct?
```

### 2. **Run Main Pipeline (Cell 2)**
Executes the full data enrichment:
- Aggregates 1M+ units into project summaries
- Merges with project metadata
- Creates 120+ output columns
- **Time**: ~5-10 minutes

### 3. **Run Category Refinement (Cells 3-6)**
Adds business-category groupings:
- Property subtypes → Flat/Shop/Office/Other
- Property type flags (Residential/Commercial/Mixed)
- Floor information
- **Time**: ~2-3 minutes

### 4. **Review Outputs**
```
✓ Main file: Dubai_Merged_Data_Project_wise_carpet_area.xlsx
  - 120+ columns describing each project
  - Ready for BI tools, dashboards, statistical analysis
  - One row per unique project with complete enrichment
```

---

# 1️⃣ Main Data Processing Pipeline
## Complete Dubai Projects Dataset Enrichment (2025)

## <a id="architecture"></a>🏗️ Pipeline Architecture

### Data Flow Diagram:
```
┌─────────────────────────────────────────────────────────────────────────┐
│                    STAGE 1: MAIN ENRICHMENT PIPELINE                   │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  Projects.csv      Units.csv     Buildings.csv                          │
│       │                │                │                               │
│       └────────────────┴────────────────┘                               │
│                      │                                                  │
│              1. NORMALIZE & DEDUPE                                      │
│              2. AGGREGATE (chunked reading)                             │
│                 ├─ Group units by project_id                           │
│                 ├─ Count rooms, areas, subtypes                        │
│                 └─ Store room-area pairs for nested dict               │
│              3. MERGE (project_id join)                                │
│              4. ENRICH (date, quarters)                                │
│              5. CATEGORIZE (room types)                                │
│              6. EXPAND (bedroom columns)                               │
│              7. CALCULATE AREAS                                        │
│              8. JSON SERIALIZATION                                     │
│                     │                                                  │
│                     ▼                                                  │
│  Dubai_Merged_Data_Project_wise_carpet_area.xlsx                       │
│  (120+ columns, 1 row per project)                                    │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────┐
│       STAGES 2-6: CATEGORY & BUSINESS LOGIC ENRICHMENT                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  Excel (from Stage 1)                                                  │
│       │                                                                  │
│       ├─→ Summarize property subtypes → Flat/Shop/Office/Other         │
│       ├─→ Classify mixed-use types → Residential/Commercial/Mixed      │
│       ├─→ Attach building floors → floor_list column                   │
│       │                                                                  │
│       └─→ Updated Excel with 4 new business columns                    │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘


```

### Processing Stages:

| Stage | Input | Output | Purpose | Time |
|-------|-------|--------|---------|------|
| **1️⃣ Aggregation** | Units.csv (chunked) | Project-level dicts | Group unit data by project | 3-8 min |
| **2️⃣ Enrichment** | Projects.csv + Agg | Enhanced projects | Merge with project metadata | 1 min |
| **3️⃣ Categorization** | Room types | Bedroom buckets | Standardize unit classifications | Instant |
| **4️⃣ Expansion** | Categories | New columns | Create individual bedroom columns | Instant |
| **5️⃣ Area Calculation** | Actual areas | Carpet area | Total sqft per area/category | Instant |
| **6️⃣ JSON Serialization** | Dictionaries | JSON strings | Store complex data structures | Instant |
| **7️⃣ Property Flagging** | Unit counts | Business flags | Residential/Commercial classification | Instant |
| **8️⃣ Floor Assignment** | Buildings.csv | Floor list | Attach building floors to projects | 1 min |

---

## Data Quality Assurances

### Handling Missing Data:
- **Missing rooms**: Tracked as "NA" in dicts (not lost)
- **Missing areas**: Tracked as "NULL" in separate count dict
- **Missing dates**: Replaced with NaN, handled in quarter calculations
- **Missing subtypes**: Shown as "NA" in property type dicts

### Type Safety:
- Project IDs normalized to consistent string format
- Float values (areas, counts) safely converted with error handling

---

## Data Quality Assurances

### Handling Missing Data:
- **Missing rooms**: Tracked as "NA" in dicts (not lost)
- **Missing areas**: Tracked as "NULL" in separate count dict
- **Missing dates**: Replaced with NaN, handled in quarter calculations
- **Missing subtypes**: Shown as "NA" in property type dicts

In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════════
  MAIN PIPELINE: Build Complete Dubai Projects Dataset with Enriched Metrics
═══════════════════════════════════════════════════════════════════════════════

ALGORITHM OVERVIEW:
1. Load & deduplicate Projects.csv
2. Stream Units.csv in 500K-row chunks to build project-level aggregations
3. Merge aggregated units back into projects table
4. Enrich with date/quarter calculations
5. Categorize room types into standardized buckets
6. Expand bedroom types into individual columns
7. Calculate cumulative actual areas
8. Create JSON dictionaries for complex room-area relationships
9. Extract carpet area per bedroom category
10. Save enriched Excel workbook

KEY FEATURES:
✓ Memory-efficient: chunked reading for large unit datasets
✓ Lossless: tracks NULL/missing values separately
✓ Flexible: handles JSON/dict/string formats safely
✓ Comprehensive: includes carpet area calculations per property type

OUTPUT COLUMNS (120+ total):
  - Base: project_id, project_name_en/ar, location info, dates
  - Units: units_rooms_en (dict), units_actual_area (dict), unit_count
  - Categories: < 1 B/R, 1 B/R, 2 B/R, ..., > 5 B/R, PENTHOUSE, Commercial, Other
  - Carpet Areas: carpet_area_lt_1_BhR, carpet_area_1_BhR, ..., per category
  - JSON: rooms_area_dict, rooms_area_dict_categorized (complex nesting)
"""

import pandas as pd
import json
import ast
import re
from collections import defaultdict

# ────────────────────────────────────────────────
# HELPER FUNCTIONS - Data Type Safety & Conversion
# ────────────────────────────────────────────────
def to_dict_safe(x):
    """
    Safely convert any input to a dict.
    Handles: native dicts, JSON strings, Python literal strings, pandas NaN, None.
    Returns: dict or {} if conversion fails.
    """
    if isinstance(x, dict):
        return x
    if x is None or (isinstance(x, float) and pd.isna(x)) or (isinstance(x, str) and x.strip() == ""):
        return {}
    if isinstance(x, str):
        s = x.strip()
        if s.lower() in {"nan", "none", "null", "{}"}:
            return {}
        try:
            obj = json.loads(s)
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass
    return {}

def categorize_units(x):
    """
    Map room types to standardized bedroom categories.
    
    LOGIC:
    - "SINGLE ROOM", "SINGLE" → < 1 B/R
    - "PENTHOUSE" → PENTHOUSE
    - GYM, HOTEL, KIOSK, OFFICE, SHOP, STUDIO → Commercial
    - Regex match: "1 BR", "2 B/R", etc. → 1 B/R, 2 B/R, ... > 5 B/R
    - Everything else → Other
    
    RETURNS: dict with category names as keys, unit counts as values
    """
    data = to_dict_safe(x)
    result = {}
    comm_kw = ['GYM', 'HOTEL', 'KIOSK', 'OFFICE', 'SHOP', 'STUDIO']

    for k, v in data.items():
        ks = str(k).strip().upper()
        try: val = float(v)
        except: val = 0

        if 'SINGLE ROOM' in ks or ks == 'SINGLE':
            result['< 1 B/R'] = result.get('< 1 B/R', 0) + val; continue
        if 'PENTHOUSE' in ks:
            result['PENTHOUSE'] = result.get('PENTHOUSE', 0) + val; continue
        if any(w in ks for w in comm_kw):
            result['Commercial'] = result.get('Commercial', 0) + val; continue

        m = re.search(r'(\d+)\s*(?:B\s*/?\s*R?|BR|BHK)\b', ks, re.IGNORECASE)
        num = int(m.group(1)) if m else None

        if num is not None:
            rk = {1:'1 B/R', 2:'2 B/R', 3:'3 B/R', 4:'4 B/R', 5:'5 B/R'}.get(num, '> 5 B/R' if num > 5 else 'Other')
            result[rk] = result.get(rk, 0) + val
        else:
            result['Other'] = result.get('Other', 0) + val

    return {k: int(v) if v == int(v) else v for k, v in result.items()}

def calculate_cumulative_area(x):
    """
    Calculate total carpet area: Σ(area_value × unit_count).
    Safely handles missing areas by skipping them.
    
    INPUT: dict of {area_sqft: count, ...}
    RETURNS: float, rounded to 2 decimals
    """
    data = to_dict_safe(x)
    total = 0.0
    for k, v in data.items():
        try: total += float(k) * float(v)
        except: continue
    return round(total, 2)

def normalize_project_id(x):
    """
    Normalize project_id to consistent string format.
    - Handles float IDs (e.g., "123.0" → "123")
    - Removes trailing ".0"
    - Returns None for empty/invalid inputs
    """
    if pd.isna(x): return None
    s = str(x).strip()
    try:
        f = float(s)
        if f.is_integer(): return str(int(f))
    except: pass
    s = re.sub(r"\.0$", "", s).strip()
    return s if s else None

def map_room_key_to_bucket(room_key: str) -> str:
    """
    Map individual room type (e.g., "2 BHK") to category bucket.
    Uses same logic as categorize_units() but for single room type.
    """
    s = str(room_key).strip().upper()

    if "SINGLE ROOM" in s or s == "SINGLE":
        return "< 1 B/R"
    if "PENTHOUSE" in s:
        return "PENTHOUSE"
    commercial_keywords = ['GYM', 'HOTEL', 'KIOSK', 'OFFICE', 'SHOP', 'STUDIO']
    if any(kw in s for kw in commercial_keywords):
        return "Commercial"

    m = re.search(r'(\d+)\s*B\s*/?\s*R\b|(\d+)\s*BR\b', s)
    if m:
        num = int(m.group(1) or m.group(2))
        if num == 1: return "1 B/R"
        if num == 2: return "2 B/R"
        if num == 3: return "3 B/R"
        if num == 4: return "4 B/R"
        if num == 5: return "5 B/R"
        if num > 5:  return "> 5 B/R"

    return "Other"

def normalize_rooms_area_dict(x):
    """
    Convert nested room→{area: count} dictionary into categorized format.
    
    INPUT:  {"2 BHK": {"500.00": 5, "600.00": 3}, ...}
    OUTPUT: {"2 B/R": {"500.00": 8}, ... grouped by bucket}
    
    LOGIC: Group original room types into standard categories, aggregate counts per area.
    """
    data = to_dict_safe(x)
    out = defaultdict(lambda: defaultdict(int))
    for outer_key, inner in data.items():
        bucket = map_room_key_to_bucket(outer_key)
        for area_k, cnt_v in to_dict_safe(inner).items():
            try:
                area_clean = f"{float(area_k):.2f}"
                count_clean = int(float(cnt_v))
                out[bucket][area_clean] += count_clean
            except:
                continue
    return {k: dict(v) for k, v in out.items()}

def sum_area_times_count(inner_dict):
    """
    Calculate total area from {area_sqft: count} dict.
    Formula: Σ(area × count)
    """
    d = to_dict_safe(inner_dict)
    total = 0.0
    for k, v in d.items():
        try:
            area = float(k)
            cnt = float(v)
            total += area * cnt
        except:
            continue
    return round(total, 2)

def extract_carpet_areas_per_bucket(x):
    """
    Extract carpet area for each room category.
    
    INPUT:  {"2 B/R": {"500.00": 5, "600.00": 3}, ...}
    OUTPUT: {
             "carpet_area_2_BhR": 4300.00,
             "carpet_area_1_BhR": 2100.00,
             ...
            }
    """
    outer = to_dict_safe(x)
    out = {}
    for bucket, inner in outer.items():
        safe_bucket = str(bucket).strip().replace(' ', '_').replace('/', '_').replace('<', 'lt').replace('>', 'gt')
        col_name = f"carpet_area_{safe_bucket}"
        out[col_name] = sum_area_times_count(inner)
    return out

# ────────────────────────────────────────────────
# MAIN FUNCTION
# ────────────────────────────────────────────────
def build_complete_dubai_dataset():
    projects_file = r"D:\Dubai\Data\Projects.csv"
    units_file    = r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Units_with_actual_area_sqft.csv"
    output_file   = r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise_carpet_area.xlsx"

    encoding = 'utf-8'

    print("1. Loading Projects...")
    df = pd.read_csv(projects_file, encoding=encoding, encoding_errors='replace')
    df = df.drop_duplicates(subset=['project_id'], keep='first')
    df['_pid_norm'] = df['project_id'].apply(normalize_project_id)

    print("2. Aggregating Units (chunked)...")
    units_columns = [
        'project_id', 'rooms_en', 'actual_area_sqft',
        'property_sub_type_en', 'project_name_en', 'project_name_ar'
    ]

    project_data = defaultdict(lambda: {
        'rooms': defaultdict(int),
        'areas': defaultdict(int),
        'subtypes': defaultdict(int),
        'name_en': None,
        'name_ar': None,
        'unit_count': 0,
        'room_area_pairs': []
    })

    chunk_size = 500_000
    chunk_num = 0

    for chunk in pd.read_csv(units_file, chunksize=chunk_size, encoding=encoding, encoding_errors='replace'):
        chunk_num += 1
        print(f"   → chunk {chunk_num}", end='\r')
        avail = [c for c in units_columns if c in chunk.columns]
        chunk = chunk[avail]
        chunk['_pid_norm'] = chunk['project_id'].apply(normalize_project_id)

        for _, row in chunk.iterrows():
            pid = row['_pid_norm']
            if pd.isna(pid): continue
            project_data[pid]['unit_count'] += 1

            room = row.get('rooms_en')
            rk = 'NA' if pd.isna(room) or str(room).strip() == '' else str(room).strip()
            project_data[pid]['rooms'][rk] += 1

            area = row.get('actual_area_sqft')
            ak = 'NA' if pd.isna(area) or str(area).strip() == '' else f"{float(area):.2f}" if area else 'NA'
            project_data[pid]['areas'][ak] += 1

            subtype = row.get('property_sub_type_en')
            sk = 'NA' if pd.isna(subtype) or str(subtype).strip() == '' else str(subtype).strip()
            project_data[pid]['subtypes'][sk] += 1

            for col, tgt in [('project_name_en','name_en'), ('project_name_ar','name_ar')]:
                v = row.get(col)
                if pd.notna(v) and project_data[pid][tgt] is None:
                    project_data[pid][tgt] = str(v).strip()

            project_data[pid]['room_area_pairs'].append((row.get('rooms_en'), row.get('actual_area_sqft')))

    print("\n3. Building aggregated table...")
    agg_rows = []
    for pid, d in project_data.items():
        nested = defaultdict(lambda: defaultdict(int))
        for r, a in d['room_area_pairs']:
            rk = "NA" if pd.isna(r) or str(r).strip() == "" else str(r).strip()
            ak = "NULL" if pd.isna(a) or str(a).strip() == "" else f"{float(a):.2f}" if a else "NULL"
            nested[rk][ak] += 1

        agg_rows.append({
            '_pid_norm': pid,
            'units_rooms_en': dict(d['rooms']),
            'units_actual_area': dict(d['areas']),
            'units_property_sub_type_en': dict(d['subtypes']),
            'units_project_name_en': d['name_en'],
            'units_project_name_ar': d['name_ar'],
            'unit_count': d['unit_count'],
            'rooms_area_dict_raw': {k: dict(v) for k, v in nested.items()}
        })

    df_agg = pd.DataFrame(agg_rows)

    print("4. Merging...")
    df = df.merge(df_agg, on='_pid_norm', how='left')

    # Cleanup
    df['unit_count'] = df['unit_count'].fillna(0).astype(int)
    for c in ['units_rooms_en', 'units_actual_area', 'units_property_sub_type_en', 'rooms_area_dict_raw']:
        df[c] = df[c].apply(lambda x: x if isinstance(x, dict) else {})

    df['units_project_name_en'] = df['units_project_name_en'].fillna('')
    df['units_project_name_ar'] = df['units_project_name_ar'].fillna('')

    # Date columns
    print("5. Date / quarter columns...")
    for col in ['project_start_date', 'project_end_date']:
        df[col] = pd.to_datetime(df[col], format='%d-%m-%Y', errors='coerce')

    df['Start_Year'] = df['project_start_date'].dt.year.astype('Int64')
    df['End_Year']   = df['project_end_date'].dt.year.astype('Int64')

    df['Start_Quarter'] = ('Q' + df['project_start_date'].dt.quarter.astype('Int64').astype(str) + ' ' +
                           df['project_start_date'].dt.year.astype('Int64').astype(str)).where(df['project_start_date'].notna(), '')

    df['End_Quarter'] = ('Q' + df['project_end_date'].dt.quarter.astype('Int64').astype(str) + ' ' +
                         df['project_end_date'].dt.year.astype('Int64').astype(str)).where(df['project_end_date'].notna(), '')

    df['Start_Quarter_Year'] = df['project_start_date'].dt.to_period('Q')
    df['End_Quarter_Year']   = df['project_end_date'].dt.to_period('Q')

    # Categorization & expansion
    print("6. Categorizing & expanding bedroom types...")
    df['categorized_units'] = df['units_rooms_en'].apply(categorize_units)

    cats = ['< 1 B/R', '1 B/R', '2 B/R', '3 B/R', '4 B/R', '5 B/R', '> 5 B/R', 'PENTHOUSE', 'Commercial', 'Other']
    expanded = df['categorized_units'].apply(pd.Series).reindex(columns=cats).fillna(0).astype(int)
    df = pd.concat([df, expanded], axis=1)

    # Cumulative actual area
    print("7. Cumulative actual area...")
    df['units_actual_area_Cumulative'] = df['units_actual_area'].apply(calculate_cumulative_area)

    # Rooms-area dicts
    print("8. Creating rooms_area_dict (JSON)...")
    df['rooms_area_dict'] = df['rooms_area_dict_raw'].apply(lambda d: json.dumps(d, ensure_ascii=False) if d else '{}')

    print("9. Creating categorized rooms_area_dict...")
    df['rooms_area_dict_categorized'] = df['rooms_area_dict_raw'].apply(normalize_rooms_area_dict)
    df['rooms_area_dict_categorized'] = df['rooms_area_dict_categorized'].apply(lambda d: json.dumps(d, ensure_ascii=False) if d else '{}')

    # New: carpet area per bucket
    print("10. Calculating carpet area per bucket...")
    carpet_expanded = df['rooms_area_dict_categorized'].apply(extract_carpet_areas_per_bucket).apply(pd.Series)
    carpet_expanded = carpet_expanded.fillna(0)
    df = pd.concat([df, carpet_expanded], axis=1)

    # Final cleanup
    df.drop(columns=['_pid_norm', 'rooms_area_dict_raw'], inplace=True, errors='ignore')

    print(f"11. Saving:\n    {output_file}")
    df.to_excel(output_file, index=False)

    print("\nSummary:")
    print(f"  Total rows: {len(df):,}")
    print(f"  Projects with units: {df['unit_count'].gt(0).sum():,}")
    print(f"  Total units: {df['unit_count'].sum():,}")
    print(f"  Total actual area: {df['units_actual_area_Cumulative'].sum():,.2f} sqft")
    print(f"  Carpet area columns added: {list(carpet_expanded.columns)}")

if __name__ == "__main__":
    build_complete_dubai_dataset()

1. Loading Projects...
2. Aggregating Units (chunked)...


C:\Users\Admin\AppData\Local\Temp\ipykernel_21168\1542033842.py:189: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(units_file, chunksize=chunk_size, encoding=encoding, encoding_errors='replace'):


C:\Users\Admin\AppData\Local\Temp\ipykernel_21168\1542033842.py:189: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(units_file, chunksize=chunk_size, encoding=encoding, encoding_errors='replace'):


C:\Users\Admin\AppData\Local\Temp\ipykernel_21168\1542033842.py:189: DtypeWarning: Columns (7,10,12,13,25,26,38,39,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(units_file, chunksize=chunk_size, encoding=encoding, encoding_errors='replace'):


C:\Users\Admin\AppData\Local\Temp\ipykernel_21168\1542033842.py:189: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(units_file, chunksize=chunk_size, encoding=encoding, encoding_errors='replace'):


C:\Users\Admin\AppData\Local\Temp\ipykernel_21168\1542033842.py:189: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(units_file, chunksize=chunk_size, encoding=encoding, encoding_errors='replace'):


   → chunk 5
3. Building aggregated table...
4. Merging...
5. Date / quarter columns...
6. Categorizing & expanding bedroom types...
7. Cumulative actual area...
8. Creating rooms_area_dict (JSON)...
9. Creating categorized rooms_area_dict...
10. Calculating carpet area per bucket...
11. Saving:
    D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise_carpet_area.xlsx

Summary:
  Total rows: 3,039
  Projects with units: 2,376
  Total units: 724,700
  Total actual area: 790,813,273.61 sqft
  Carpet area columns added: ['carpet_area_1_B_R', 'carpet_area_2_B_R', 'carpet_area_3_B_R', 'carpet_area_Commercial', 'carpet_area_Other', 'carpet_area_4_B_R', 'carpet_area_5_B_R', 'carpet_area_PENTHOUSE', 'carpet_area_gt_5_B_R', 'carpet_area_lt_1_B_R']


In [ ]:
# def classify_property_type(x):
#     if not isinstance(x, dict) or len(x) == 0:
#         return None  # or "" if you prefer

#     has_residential = False
#     has_commercial = False

#     for key in x.keys():
#         key_u = key.upper()

#         # Residential keys
#         if 'BHK' in key_u or 'PENTHOUSE' in key_u:
#             has_residential = True

#         # Commercial key
#         if key_u == 'COMMERCIAL':
#             has_commercial = True

#     if has_residential and has_commercial:
#         return 'Commercial + Residential'
#     elif has_commercial:
#         return 'Commercial'
#     elif has_residential:
#         return 'Residential'
#     else:
#         return None


# df['property_type_flag'] = df['categorized_units'].apply(classify_property_type)


In [24]:
'''
Carpet area on the basis of property sub type by mapping the property types to categorized buckets like FLAT, UNIT, BUILDING to Flat, 
SHOP, SHOW ROOMS, STORE to Shop, CLINIC, OFFICE, WAREHOUSE, WORKSHOP to Office and rest to Other.


'''


import pandas as pd
import ast
import json
from collections import defaultdict

df  = pd.read_excel(r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise_carpet_area.xlsx")

import pandas as pd
import ast
import json

def to_dict_safe_lossless(x):
    """
    Always return a dict (no data loss for valid dicts/strings).
    If missing/invalid -> {}  (so it does NOT add 1 to Other)
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {}

    if isinstance(x, dict):
        return x

    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() in {"nan", "none", "null", "{}"}:
            return {}

        try:
            obj = json.loads(s)
            return obj if isinstance(obj, dict) else {}
        except Exception:
            pass

        try:
            obj = ast.literal_eval(s)
            return obj if isinstance(obj, dict) else {}
        except Exception:
            return {}

    return {}


# ---- YOUR GROUPS (case-insensitive) ----
flat_set   = {"FLAT", "UNIT", "BUILDING"}
shop_set   = {"SHOP", "SHOW ROOMS", "STORE"}
office_set = {"CLINIC", "OFFICE", "WAREHOUSE", "WORKSHOP"}

def summarize_4cols(x):
    d = to_dict_safe_lossless(x)

    out = {"Flat": 0, "Shop": 0, "Office": 0, "Other": 0}

    for k, v in d.items():
        key = str(k).strip().upper()

        # numeric safety
        try:
            cnt = int(float(v))
        except Exception:
            cnt = 0

        if key in flat_set:
            out["Flat"] += cnt
        elif key in shop_set:
            out["Shop"] += cnt
        elif key in office_set:
            out["Office"] += cnt
        else:
            out["Other"] += cnt

    return out

# ---- APPLY (creates 4 columns only) ----
four_cols = df["units_property_sub_type_en"].apply(summarize_4cols).apply(pd.Series)

df = pd.concat([df, four_cols], axis=1)

# Now df has: Flat, Shop, Office, Other
print(df[["Flat", "Shop", "Office", "Other"]].head())


   Flat  Shop  Office  Other  Other
0   583     2       0      0      0
1    76     1       0      0     74
2   230    25     299    324      0
3   158     8       0      0      0
4     0     5     260      0      0



---

# 2️⃣ Property Sub-Type Classification
## Normalize Property Types into Business Categories

**Objective**: Map detailed property sub-types into 4 high-level business categories

### Mapping Logic:
- **Flat**: FLAT, UNIT, BUILDING → Residential apartments/units
- **Shop**: SHOP, SHOW ROOMS, STORE → Retail/commercial spaces  
- **Office**: CLINIC, OFFICE, WAREHOUSE, WORKSHOP → Office/industrial uses
- **Other**: All remaining types

### Processing:
1. Load the Excel file with enriched project data
2. Parse `units_property_sub_type_en` column (dict with sub-type: count)
3. Group each sub-type into its corresponding business category
4. Sum unit counts per category
5. Create 4 new columns: Flat, Shop, Office, Other

### Output:
- New columns: `Flat`, `Shop`, `Office`, `Other` (integer counts)
- Example: A project with 150 FLATs, 50 SHOPs → Flat=150, Shop=50, Office=0, Other=0


In [27]:
df.to_excel(r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise_carpet_area.xlsx", index=False)

In [33]:
'''

   Build {property_sub_type_en_or_NA: cumulative_actual_area}
    - property_sub_type_en missing/blank -> "NA"
    - actual_area missing/invalid -> counted in "NULL" bucket (as key) so nothing is lost
      (Because numeric cumulative can't include NULL, we store NULL count separately)
    Output example:
      {
        "Flat": 12345.67,
        "Shop": 543.21,
        "NA": 100.0,
        "__NULL_COUNT__": {"Flat": 2, "Shop": 1, "NA": 5}

'''



import pandas as pd
import json
import re
from collections import defaultdict

# ---------------- CONFIG ----------------
units_csv = r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Units_with_actual_area_sqft.csv"
excel_file = r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise_carpet_area.xlsx"
out_file = r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise_carpet_area.xlsx"

join_key_units = "project_id"   # Units.csv
join_key_excel = "project_id"   # Excel

# -------------- HELPERS ----------------
def norm_project_id(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass
    s = re.sub(r"\.0$", "", s).strip()
    return s if s else None

def subtype_area_dict_lossless(group: pd.DataFrame) -> dict:
    """
    LOSSLESS:
    Build {property_sub_type_en_or_NA: cumulative_actual_area}
    - property_sub_type_en missing/blank -> "NA"
    - actual_area missing/invalid -> counted in "NULL" bucket (as key) so nothing is lost
      (Because numeric cumulative can't include NULL, we store NULL count separately)
    Output example:
      {
        "Flat": 12345.67,
        "Shop": 543.21,
        "NA": 100.0,
        "__NULL_COUNT__": {"Flat": 2, "Shop": 1, "NA": 5}
      }
    """
    area_sum = defaultdict(float)
    null_count = defaultdict(int)

    for subtype, area in zip(group["property_sub_type_en"], group["actual_area_sqft"]):
        # subtype key
        if pd.isna(subtype) or str(subtype).strip() == "":
            sub_key = "NA"
        else:
            sub_key = str(subtype).strip()

        # area
        if pd.isna(area) or str(area).strip() == "":
            null_count[sub_key] += 1
            continue

        try:
            area_val = float(area)
            area_sum[sub_key] += area_val
        except Exception:
            null_count[sub_key] += 1

    out = {k: round(v, 2) for k, v in area_sum.items()}

    # keep NULL info so nothing is lost
    if len(null_count) > 0:
        out["__NULL_COUNT__"] = dict(null_count)

    return out

# ---------------- LOAD ----------------
df_units = pd.read_csv(units_csv, low_memory=False)
df_xl = pd.read_excel(excel_file)

# ---------------- VALIDATE ----------------
required_units_cols = {join_key_units, "property_sub_type_en", "actual_area_sqft"}
missing_units = required_units_cols - set(df_units.columns)
if missing_units:
    raise ValueError(f"Units.csv missing columns: {missing_units}\nAvailable: {list(df_units.columns)}")

if join_key_excel not in df_xl.columns:
    raise ValueError(f"Excel missing column: {join_key_excel}\nAvailable: {list(df_xl.columns)}")

# ---------------- NORMALIZE JOIN KEY ----------------
df_units["_pid"] = df_units[join_key_units].apply(norm_project_id)
df_xl["_pid"] = df_xl[join_key_excel].apply(norm_project_id)

# ---------------- AGGREGATE ----------------
df_units_small = df_units.loc[df_units["_pid"].notna(), ["_pid", "property_sub_type_en", "actual_area_sqft"]].copy()

df_subtype_area = (
    df_units_small
    .groupby("_pid", dropna=False)
    .apply(subtype_area_dict_lossless)
    .reset_index(name="subtype_actual_area_dict")
)

# Store dict as JSON string for Excel
df_subtype_area["subtype_actual_area_dict"] = df_subtype_area["subtype_actual_area_dict"].apply(
    lambda d: json.dumps(d, ensure_ascii=False)
)

# ---------------- MERGE INTO EXCEL ----------------
df_out = df_xl.merge(df_subtype_area, on="_pid", how="left")
df_out["subtype_actual_area_dict"] = df_out["subtype_actual_area_dict"].fillna("{}")

# ---------------- SAVE ----------------
df_out.drop(columns=["_pid"], inplace=True)
df_out.to_excel(out_file, index=False)

print("Done. Saved:", out_file)
print("New column added: subtype_actual_area_dict")


C:\Users\Admin\AppData\Local\Temp\ipykernel_21168\1606689769.py:112: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(subtype_area_dict_lossless)


Done. Saved: D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise_carpet_area.xlsx
New column added: subtype_actual_area_dict



---

# 3️⃣ Property Sub-Type Cumulative Area Calculation
## Calculate Total Carpet Area by Property Sub-Type

**Objective**: Create a lossless aggregation of actual area grouped by property sub-type

### Algorithm:
1. **Read Units CSV** (chunked for memory efficiency)
2. **Group by project_id**:
   - For each project, collect all (property_sub_type_en, actual_area_sqft) pairs
   - Handle missing subtypes as "NA"
   - Handle missing areas as NULL counts (tracked separately)
3. **Calculate cumulative areas**: Σ(actual_area_sqft) per subtype per project
4. **Create JSON output**: 
   ```json
   {
     "Flat": 12345.67,
     "Shop": 543.21,
     "NA": 100.0,
     "__NULL_COUNT__": {"Flat": 2, "Shop": 1, "NA": 5}
   }
   ```
5. **Merge back** to main Excel workbook

### Key Features:
- ✓ **Lossless**: Tracks NULL values in separate `__NULL_COUNT__` dict
- ✓ **Robust**: Handles float/string conversions safely
- ✓ **Scalable**: Works with large CSV files via iteration
- ✓ **JSON compatible**: Stores in Excel as JSON strings

### Output:
- New column: `subtype_actual_area_dict` (JSON string)
- Example value for Flat+Shop project: `{"Flat": 150000.00, "Shop": 45320.50}`


In [32]:
dfdd

,project_id,project_number,project_name,developer_id,developer_number,developer_name,master_developer_id,master_developer_number,master_developer_name,project_start_date,...,carpet_area_Other,carpet_area_4_B_R,carpet_area_5_B_R,carpet_area_PENTHOUSE,carpet_area_gt_5_B_R,carpet_area_lt_1_B_R,Flat,Shop,Office,Other
0,85,85,سنتريوم تاور,26.0,26.0,تيكوم للإستثمارات منطقة حرة- ذ.م.م,856,856,تيكوم للإستثمارات منطقة حرة- ذ.م.م,2006-07-11,...,0.00,0.00,0.00,0.0,0.0,0.0,583,2,0,0
1,9533811,1364,داماك هيلز - جولف فيدوتا,8289244.0,954.0,داماك كريسنت للعقارات (ش.ذ.م.م),8289244,954,داماك كريسنت للعقارات (ش.ذ.م.م),2013-08-01,...,0.00,0.00,0.00,0.0,0.0,0.0,76,1,0,74
2,102,102,دبي ستار,29414636.0,1132.0,مركز دبي للسلع المتعددة,153,153,مركز دبي للسلع المتعددة,2006-11-01,...,275313.89,19558.45,0.00,0.0,0.0,0.0,230,25,299,0
3,42,42,السيف I,15.0,15.0,اعمار العقارية (ش . م. ع),555,555,اعمار العقارية (ش . م. ع),2003-05-17,...,0.00,15213.05,106482.32,0.0,0.0,0.0,158,8,0,0
4,219,219,سلفر تاور,99.0,99.0,دبي للعقارات (ش.ذ.م.م),850,850,دبي للعقارات (ش.ذ.م.م),2007-12-03,...,0.00,0.00,0.00,0.0,0.0,0.0,0,5,260,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3034,771287785,3826,كينجدوم غيت,715072308.0,2219.0,الفرجان ( ش.ذ.م.م ),121,121,الفرجان ( ش.ذ.م.م ),2025-04-01,...,2363.44,0.00,0.00,0.0,0.0,0.0,82,4,0,0
3035,772020432,3828,أفينيو 888- مودو,452615258.0,1511.0,دبي الجنوب للعقارات دي دبليو سي ش.ذ.م.م,31550449,1142,دبي الجنوب للعقارات دي دبليو سي ش.ذ.م.م,2025-11-15,...,16072.23,0.00,0.00,0.0,0.0,0.0,217,6,0,1
3036,757449726,3752,فياني باي ميتيورا,582153472.0,1841.0,ليوان(ش.ذ.م.م.),27962557,1121,ليوان(ش.ذ.م.م.),2025-06-01,...,4565.94,0.00,0.00,0.0,0.0,0.0,117,1,0,0
3037,751875646,3725,بن غاطي سكاي بليد,15782130.0,1051.0,اعمار العقارية (ش . م. ع),555,555,اعمار العقارية (ش . م. ع),2025-02-01,...,5906.80,0.00,0.00,0.0,0.0,0.0,594,2,0,0


In [37]:
''''
    Input:
      {"Flat": 1200, "Office": 800, "Hotel": 300, "__NULL_COUNT__": {...}}

    Output:
      {
        "carpet_area_Flat": 1200,
        "carpet_area_Shop": 0,
        "carpet_area_Office": 800,
        "carpet_area_Other": 300
      }


'''


import pandas as pd
import ast
import json

# ---------------- LOAD ----------------
df = pd.read_excel(
    r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise_carpet_area.xlsx"
)
AREA_DICT_COL = "subtype_actual_area_dict"  # must exist in file

# ---------------- SAFE DICT PARSER ----------------
def to_dict_safe(x):
    if isinstance(x, dict):
        return x
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {}
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() in {"nan", "none", "null", "{}"}:
            return {}
        try:
            return json.loads(s)
        except Exception:
            try:
                return ast.literal_eval(s)
            except Exception:
                return {}
    return {}

# ---------------- GROUP DEFINITIONS ----------------
flat_set   = {"FLAT", "UNIT", "BUILDING"}
shop_set   = {"SHOP", "SHOW ROOMS", "STORE"}
office_set = {"CLINIC", "OFFICE", "WAREHOUSE", "WORKSHOP"}

# ---------------- CORE FUNCTION ----------------
def summarize_4cols_carpet_area(x):
    """
    Input:
      {"Flat": 1200, "Office": 800, "Hotel": 300, "__NULL_COUNT__": {...}}

    Output:
      {
        "carpet_area_Flat": 1200,
        "carpet_area_Shop": 0,
        "carpet_area_Office": 800,
        "carpet_area_Other": 300
      }
    """
    d = to_dict_safe(x)

    out = {
        "carpet_area_Flat": 0.0,
        "carpet_area_Shop": 0.0,
        "carpet_area_Office": 0.0,
        "carpet_area_Other": 0.0
    }

    for k, v in d.items():
        key_u = str(k).strip().upper()

        # ignore meta buckets safely
        if key_u in {"__NULL_COUNT__", "NULL", "NA"}:
            out["carpet_area_Other"] += 0
            continue

        try:
            area_val = float(v)
        except Exception:
            area_val = 0.0

        if key_u in flat_set:
            out["carpet_area_Flat"] += area_val
        elif key_u in shop_set:
            out["carpet_area_Shop"] += area_val
        elif key_u in office_set:
            out["carpet_area_Office"] += area_val
        else:
            out["carpet_area_Other"] += area_val

    # round for reporting
    for c in out:
        out[c] = round(out[c], 2)

    return out

# ---------------- APPLY ----------------
if AREA_DICT_COL not in df.columns:
    raise ValueError(f"Column '{AREA_DICT_COL}' not found. Available columns: {list(df.columns)}")

carpet_cols = df[AREA_DICT_COL].apply(summarize_4cols_carpet_area).apply(pd.Series)

df = pd.concat([df, carpet_cols], axis=1)

print(df[[
    "carpet_area_Flat",
    "carpet_area_Shop",
    "carpet_area_Office",
    "carpet_area_Other"
]].head())

# ---------------- SAVE ----------------
out_file = (
    r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise_carpet_area.xlsx"
)
df.to_excel(out_file, index=False)

print("Saved:", out_file)


   carpet_area_Flat  carpet_area_Shop  carpet_area_Office  carpet_area_Other  \
0         571198.84           4473.05                0.00               0.00   
1         109502.98            608.05                0.00               0.00   
2         190887.00          23473.16           251841.14          275313.89   
3         560127.31           8209.41                0.00               0.00   
4              0.00          10970.89           267562.89               0.00   

   carpet_area_Other  
0               0.00  
1           66218.97  
2               0.00  
3               0.00  
4               0.00  
Saved: D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise_carpet_area.xlsx



---

# 4️⃣ Extract Carpet Area per Business Category
## Convert Detailed Sub-Types into 4 Business Metrics

**Objective**: Transform `subtype_actual_area_dict` (detailed property subtypes) into 4 carpet area columns

### Processing Logic:
```
Input (subtype_actual_area_dict):
{
  "Flat": 1200,
  "Office": 800,
  "Hotel": 300,
  "NA": 50,
  "__NULL_COUNT__": {...}
}
        ↓ (GROUP BY BUSINESS CATEGORY)
        ↓
Output (4 new columns):
{
  "carpet_area_Flat": 1200,
  "carpet_area_Shop": 0,
  "carpet_area_Office": 800,
  "carpet_area_Other": 350  ← Hotel (300) + NA (50)
}
```

### Category Mapping:
- **carpet_area_Flat** ← Flat
- **carpet_area_Shop** ← Shop, Show Rooms, Store
- **carpet_area_Office** ← Clinic, Office, Warehouse, Worksheet
- **carpet_area_Other** ← Everything else (Hotel, NA, etc.)

### Safe Parsing:
- Handles JSON/dict/string formats
- Skips NULL and meta buckets (`__NULL_COUNT__`)
- Defaults missing categories to 0.0
- Rounds to 2 decimal places

### Output:
- 4 new numeric columns added to dataframe
- All values are float (carpet area in sqft)
- Filled with 0 for missing/null values


In [36]:
df.sample()

,project_id,project_number,project_name,developer_id,developer_number,developer_name,master_developer_id,master_developer_number,master_developer_name,project_start_date,...,carpet_area_lt_1_B_R,Flat,Shop,Office,Other.1,subtype_actual_area_dict,carpet_area_Flat,carpet_area_Shop,carpet_area_Office,carpet_area_Other
208,376764714,2295,عزيزي ريفييرا 45,12595870.0,1002.0,مجموعة ميدان (ش.ذ.م.م),53873272,1181,مجموعة ميدان (ش.ذ.م.م),2020-11-23,...,0.0,177,13,0,0,"{""Flat"": 85153.43, ""Shop"": 9351.46}",85153.43,9351.46,0.0,0.0


In [38]:
'''
property type flag based on the presence of area in the categorized buckets (Commercial, Residential, Residential + Commercial) by checking the area values in the respective columns.


'''


import numpy as np

# Columns to use
cols = ["Flat", "Shop", "Office", "Other.1"]

# make sure numeric + no NaN
for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

def property_flag(row):
    flat = row["Flat"] > 0
    shop = row["Shop"] > 0
    office = row["Office"] > 0
    other = row["Other.1"] > 0

    # only flat > 0 and rest 0
    if flat and (not shop) and (not office) and (not other):
        return "Residential"

    # if flat > 0 AND any of (shop/office/other) > 0
    if flat and (shop or office or other):
        return "Residential + Commercial"

    # if flat == 0 AND any of (shop/office/other) > 0
    if (not flat) and (shop or office or other):
        return "Commercial"

    # all zero
    return "NA"

df["property_type_flag"] = df.apply(property_flag, axis=1)



---

# 5️⃣ Property Type Classification
## Determine Residential vs Commercial vs Mixed Projects

**Objective**: Classify each project based on the presence/absence of Flat and Commercial units

### Classification Logic:

| Condition | Output |
|-----------|--------|
| Flat > 0 AND (Shop=0, Office=0, Other=0) | **Residential** |
| Flat > 0 AND (Shop > 0 OR Office > 0 OR Other > 0) | **Residential + Commercial** |
| Flat = 0 AND (Shop > 0 OR Office > 0 OR Other > 0) | **Commercial** |
| All = 0 | **NA** |

### Decision Tree:
```
Has Residential (Flat > 0)?
├─ YES
│  └─ Has Commercial (Shop/Office/Other > 0)?
│     ├─ YES → "Residential + Commercial"
│     └─ NO  → "Residential"
└─ NO
   └─ Has Commercial?
      ├─ YES → "Commercial"
      └─ NO  → "NA"
```

### Output:
- New column: `property_type_flag`
- Values: "Residential", "Residential + Commercial", "Commercial", "NA"
- Useful for: Portfolio segmentation, market analysis, investment strategy

### Notes:
- Based on COUNTED UNITS, not property diversity in name/types
- "Residential + Commercial" = Mixed-use complex
- Can be used for location analysis, investment classification, portfolio segmentation


In [39]:
''' Adding a new column "floor_list" to the main dataframe that contains a list of floor counts for each project by 
merging the building data based on project_id and grouping the floor counts into a list.
'''


import pandas as pd

# Load files
df = pd.read_excel(r"D:\Dubai\Dubai_Merged_Data_Project_wise_Final_withFloors.xlsx")
df_buidling = pd.read_csv(r"D:\Dubai\Data\Buildings.csv")

# Ensure project_id types match (very important for merge)
df["project_id"] = pd.to_numeric(df["project_id"], errors="coerce").astype("Int64")
df_buidling["project_id"] = pd.to_numeric(df_buidling["project_id"], errors="coerce").astype("Int64")

# Convert floors -> numeric, replace NaN with 0, convert to int
df_buidling["floors"] = pd.to_numeric(df_buidling["floors"], errors="coerce")
df_buidling["floor_int"] = df_buidling["floors"].fillna(0).round().astype(int)

# Group floors list by project_id (keeps 0s)
floors_by_project = (
    df_buidling.groupby("project_id")["floor_int"]
    .apply(list)
    .reset_index(name="floor_list")
)

# Merge into main df
df = df.merge(floors_by_project, on="project_id", how="left")

# If a project_id has no building rows at all, floor_list will be NaN -> set to empty list (optional)
df["floor_list"] = df["floor_list"].apply(lambda x: x if isinstance(x, list) else [])

print(df[["project_id", "floor_list"]].head(10))

# Quick check for project_id = 102
print(df.loc[df["project_id"] == 102, ["project_id", "floor_list"]].head(5))


C:\Users\Admin\AppData\Local\Temp\ipykernel_21168\1183070752.py:10: DtypeWarning: Columns (35,37,38,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df_buidling = pd.read_csv(r"D:\Dubai\Data\Buildings.csv")


   project_id floor_list
0        1102         []
1          93         []
2    40554326         []
3   390512999         []
4   390527306         []
5   202099775         []
6   447563473         []
7   630326468         []
8   605547358         []
9   717667526         []
      project_id  floor_list
2246         102  [46, 0, 0]



---

# 6️⃣ Building Floor Information Enrichment
## Attach Building Floors Per Project

**Objective**: Merge Buildings.csv to add floor information for each project

### Process:
1. **Load files**:
   - Main dataframe (Excel with enriched project data)
   - Buildings.csv (building-level records with floor counts)

2. **Normalize project_id**: Ensure consistent int64 type for reliable merge

3. **Process floors**:
   - Convert to numeric (handle missing values)
   - Replace NaN with 0
   - Round to integers

4. **Group by project_id**:
   - Collect all floor counts per project
   - Create list: [3, 5, 2, 7, ...] 
   - Preserves 0 values (incomplete buildings)

5. **Merge**: Left join to keep all projects (even those without buildings)

6. **Cleanup**: Replace NaN floor_list with empty list []

### Output:
- New column: `floor_list` (list of integers)
- Example: `[10, 15, 20, 25, 30]` for a 5-building project
- Useful for: Building complexity analysis, structural assessment, height profiling

### Notes:
- ✓ Projects may have 0, 1, or multiple buildings
- ✓ Building may have 0 floors recorded (incomplete/ongoing)
- ✓ Preserves all values as-is for later analysis
- ✓ Can compute statistics: max floors, avg floors, total floors per project


In [40]:
df.to_excel(r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise.xlsx", index=False)



---

# 🔧 Advanced Usage & Analysis Examples

## Post-Generation Analysis Code

After running the full pipeline, use these examples for common analyses:

### Load the Output Files:
```python
import pandas as pd
import json

# Main dataset
df = pd.read_excel(r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise_carpet_area.xlsx")

# Optional: Load JSON dicts
df['rooms_area_dict_parsed'] = df['rooms_area_dict'].apply(lambda x: json.loads(x) if isinstance(x, str) else {})
```

### Analysis 1: Top Projects by Unit Count
```python
# Get top 20 projects
top_projects = df.nlargest(20, 'unit_count')[
    ['project_id', 'project_name_en', 'area_name_en', 'unit_count', 'units_actual_area_Cumulative']
]
top_projects.to_csv("top_20_projects.csv", index=False)
```

### Analysis 2: Market Segmentation
```python
# Count projects by property type
property_breakdown = df['property_type_flag'].value_counts()
print("Projects by Type:")
print(f"  Residential: {property_breakdown.get('Residential', 0):,}")
print(f"  Residential + Commercial: {property_breakdown.get('Residential + Commercial', 0):,}")
print(f"  Commercial: {property_breakdown.get('Commercial', 0):,}")

# Unit distribution
residential_units = df[df['property_type_flag'] == 'Residential']['unit_count'].sum()
commercial_units = df[df['property_type_flag'] == 'Commercial']['unit_count'].sum()
mixed_units = df[df['property_type_flag'] == 'Residential + Commercial']['unit_count'].sum()

print(f"\nUnits by Type:")
print(f"  Residential: {residential_units:,}")
print(f"  Commercial: {commercial_units:,}")
print(f"  Mixed: {mixed_units:,}")
print(f"  Total: {residential_units + commercial_units + mixed_units:,}")
```

### Analysis 3: Bedroom Type Distribution
```python
# Total units per bedroom type across all projects
bedroom_cols = ['< 1 B/R', '1 B/R', '2 B/R', '3 B/R', '4 B/R', '5 B/R', '> 5 B/R', 'PENTHOUSE', 'Commercial', 'Other']
bedroom_totals = df[bedroom_cols].sum()

print("Units by Bedroom Type:")
for br_type, count in bedroom_totals.items():
    pct = (count / bedroom_totals.sum()) * 100
    print(f"  {br_type:15s}: {count:8,} ({pct:5.1f}%)")
```


---

# 🔧 Advanced Usage & Analysis Examples

## Post-Generation Analysis Code

After running the full pipeline, use these examples for common analyses:

### Load the Output Files:
```python
import pandas as pd
import json

# Main dataset
df = pd.read_excel(r"D:\Dubai\Dubai_Updated_data\Project_Unit_Summary\Dubai_Merged_Data_Project_wise_carpet_area.xlsx")

# Optional: Load JSON dicts
df['rooms_area_dict_parsed'] = df['rooms_area_dict'].apply(lambda x: json.loads(x) if isinstance(x, str) else {})
```

### Analysis 1: Top Projects by Unit Count
```python
# Get top 20 projects
top_projects = df.nlargest(20, 'unit_count')[
    ['project_id', 'project_name_en', 'area_name_en', 'unit_count', 'units_actual_area_Cumulative']
]
top_projects.to_csv("top_20_projects.csv", index=False)
```

### Analysis 2: Market Segmentation
```python
# Count projects by property type
property_breakdown = df['property_type_flag'].value_counts()
print("Projects by Type:")
print(f"  Residential: {property_breakdown.get('Residential', 0):,}")
print(f"  Residential + Commercial: {property_breakdown.get('Residential + Commercial', 0):,}")
print(f"  Commercial: {property_breakdown.get('Commercial', 0):,}")

# Unit distribution
residential_units = df[df['property_type_flag'] == 'Residential']['unit_count'].sum()
commercial_units = df[df['property_type_flag'] == 'Commercial']['unit_count'].sum()
mixed_units = df[df['property_type_flag'] == 'Residential + Commercial']['unit_count'].sum()

print(f"\nUnits by Type:")
print(f"  Residential: {residential_units:,}")
print(f"  Commercial: {commercial_units:,}")
print(f"  Mixed: {mixed_units:,}")
print(f"  Total: {residential_units + commercial_units + mixed_units:,}")
```

### Analysis 3: Bedroom Type Distribution
```python
# Total units per bedroom type across all projects
bedroom_cols = ['< 1 B/R', '1 B/R', '2 B/R', '3 B/R', '4 B/R', '5 B/R', '> 5 B/R', 'PENTHOUSE', 'Commercial', 'Other']
bedroom_totals = df[bedroom_cols].sum()

print("Units by Bedroom Type:")
for br_type, count in bedroom_totals.items():
    pct = (count / bedroom_totals.sum()) * 100
    print(f"  {br_type:15s}: {count:8,} ({pct:5.1f}%)")
```

### Analysis 4: Location Performance
```python
# Metrics by location
location_summary = df.groupby('area_name_en').agg({
    'unit_count': 'sum',
    'units_actual_area_Cumulative': 'sum',
    'project_id': 'count'
}).rename(columns={'project_id': 'project_count'})

location_summary['avg_units_per_project'] = (location_summary['unit_count'] / location_summary['project_count']).round(0)
location_summary = location_summary.sort_values('unit_count', ascending=False)

print("Top 10 Locations by Unit Count:")
print(location_summary.head(10).to_string())
```

### Analysis 5: Analyze Projects by Start Year
```python
# Projects by start year
yearly_stats = df.groupby('Start_Year').agg({
    'unit_count': 'sum',
    'project_id': 'count',
    'units_actual_area_Cumulative': 'sum'
}).rename(columns={'project_id': 'project_count'})

print("Projects by Start Year:")
print(yearly_stats.to_string())
```

### Analysis 6: Analyze Projects by Location
```python
# Property breakdown by location
location = "Downtown Dubai"  # Change as needed

location_projects = df[df['area_name_en'] == location]
print(f"Property Type Distribution in {location}:")
print(location_projects[['project_name_en', 'unit_count', 'Flat', 'Shop', 'Office', 'Other']].head(10).to_string())
```

### Analysis 7: Carpet Area Analytics
```python
# Compare carpet area per property type
carpet_cols = ['carpet_area_Flat', 'carpet_area_Shop', 'carpet_area_Office', 'carpet_area_Other']
total_carpet = df[carpet_cols].sum()

print("Total Carpet Area by Property Type:")
for col, area in total_carpet.items():
    pct = (area / total_carpet.sum()) * 100
    print(f"  {col:25s}: {area:15,.2f} sqft ({pct:5.1f}%)")

# Average area per unit by bedroom type
bedroom_carpet = ['carpet_area_1_BhR', 'carpet_area_2_BhR', 'carpet_area_3_BhR', 'carpet_area_4_BhR']
bedroom_units = ['1 B/R', '2 B/R', '3 B/R', '4 B/R']

for carpet_col, unit_col in zip(bedroom_carpet, bedroom_units):
    if carpet_col in df.columns:
        total_area = df[carpet_col].sum()
        total_units = df[unit_col].sum()
        if total_units > 0:
            avg_area = total_area / total_units
            print(f"{unit_col}: {avg_area:.2f} sqft/unit")
```

### Analysis 8: Mixed-Use Development Analysis
```python
# Find mixed-use projects
mixed_use = df[df['property_type_flag'] == 'Residential + Commercial']

print(f"Mixed-Use Projects: {len(mixed_use)}")
print(f"Mixed-Use Units: {mixed_use['unit_count'].sum():,}")
print(f"Mixed-Use Area: {mixed_use['units_actual_area_Cumulative'].sum():,.2f} sqft")

# Breakdown of residential vs commercial in mixed projects
mixed_use[['project_name_en', 'Flat', 'Shop', 'Office']].head(10).to_string()
```

### Analysis 9: Building Complexity Analysis
```python
# Projects with building information
df_with_floors = df[df['floor_list'].apply(len) > 0].copy()
df_with_floors['max_floors'] = df_with_floors['floor_list'].apply(max)
df_with_floors['num_buildings'] = df_with_floors['floor_list'].apply(len)
df_with_floors['avg_floors'] = df_with_floors['floor_list'].apply(lambda x: sum(x) / len(x) if x else 0)

print("Building Complexity Analysis:")
print(df_with_floors[['project_name_en', 'num_buildings', 'max_floors', 'avg_floors']].describe().to_string())
```

### Analysis 10: Time-to-Completion Analysis
```python
# Projects with both start and end dates
completed = df[(df['Start_Year'].notna()) & (df['End_Year'].notna())].copy()
completed['duration_years'] = completed['End_Year'] - completed['Start_Year']

print("Project Duration Statistics:")
print(completed['duration_years'].describe())
print(f"\nLongest projects:")
print(completed.nlargest(5, 'duration_years')[['project_name_en', 'Start_Year', 'End_Year', 'duration_years']])
```

---

## 💡 Tips & Best Practices

### 🎯 For Real Estate Analysts:
- Filter by `property_type_flag` for portfolio segmentation
- Compare `units_actual_area_Cumulative` with `unit_count` for density metrics
- Analyze projects by `area_name_en` for market concentration

### 📊 For Data Scientists:
- Leverage **JSON columns** (`rooms_area_dict_categorized`) for advanced segmentation
- Use **floor_list** for structural complexity analysis
- Build predictive models using `units_actual_area_Cumulative` as target

### 📈 For BI/Dashboard Developers:
- Use **main dataset** for dimensional analysis (projects × locations × types)
- Create drill-down hierarchies: Location → Property Type → Bedroom Distribution → Project
- Build visualizations around carpet area metrics per property type

### 🔍 For Quality Assurance:
- Cross-check that `Flat+Shop+Office+Other` equals `unit_count`
- Verify that sum of individual bedroom types = `unit_count`
- Confirm JSON parse-ability: `json.loads(rooms_area_dict)` should not error

---

## ⚠️ Important Notes

### Data Interpretation:
- **unit_count**: Total units included in units.csv matching this project
- **units_actual_area_Cumulative**: Sum of (area × unit count) - not unique area
- **property_type_flag**: Based on COUNTS, not on field names
- **floor_list**: May contain 0 for incomplete buildings

### Limitations:
- Relies on project_id matching between files (no fuzzy matching)
- Assumes Units.csv is clean and consistent
- Floor information only for buildings in Buildings.csv

### Performance:
- Full pipeline: ~10-12 minutes for 1M+ units
- Excel file size: ~50-150 MB (depending on data volume)